In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Mounting google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
accepted = pd.read_csv('/content/drive/MyDrive/accepted_loans.csv', low_memory=False)

In [ ]:
print(accepted.shape)
accepted.head()

(2260701, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
accepted['loan_status'].value_counts()

,count
loan_status,
Fully Paid,1076751
Current,878317
Charged Off,268559
Late (31-120 days),21467
In Grace Period,8436
Late (16-30 days),4349
Does not meet the credit policy. Status:Fully Paid,1988
Does not meet the credit policy. Status:Charged Off,761
Default,40


In [ ]:
accepted = accepted[
    accepted['loan_status'].isin(['Fully Paid', 'Charged Off'])
].copy()

accepted['default_flag'] = accepted['loan_status'].map({
    'Fully Paid': 0,
    'Charged Off': 1
})

print(accepted['default_flag'].value_counts(normalize=True))


default_flag
0    0.800374
1    0.199626
Name: proportion, dtype: float64


In [ ]:
leakage_cols = [
    'loan_status',
    'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt',
    'next_pymnt_d', 'last_credit_pull_d'
]

accepted.drop(columns=leakage_cols, inplace=True, errors='ignore')


In [ ]:
features = [
    'loan_amnt',
    'term',
    'int_rate',
    'installment',
    'grade',
    'sub_grade',
    'emp_length',
    'home_ownership',
    'annual_inc',
    'verification_status',
    'purpose',
    'dti',
    'fico_range_low',
    'fico_range_high',
    'delinq_2yrs',
    'inq_last_6mths',
    'open_acc',
    'pub_rec',
    'revol_bal',
    'revol_util',
    'total_acc'
]

df = accepted[features + ['default_flag']].copy()


In [ ]:
# interest rate
df['int_rate'] = (
    df['int_rate'].astype(str)
    .str.replace('%', '', regex=False)
    .astype(float)
)

# term
df['term'] = df['term'].str.extract(r'(\d+)').astype(int)

# employment length
df['emp_length'] = (
    df['emp_length']
    .str.extract(r'(\d+)')
    .fillna(0)
    .astype(int)
)

# revol util
df['revol_util'] = pd.to_numeric(
    df['revol_util'].astype(str).str.replace('%', '', regex=False),
    errors='coerce'
)

df.dropna(inplace=True)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X = df.drop('default_flag', axis=1)
y = df['default_flag']

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

scaler = StandardScaler()
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

logit = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

logit.fit(X_train, y_train)

pred = logit.predict_proba(X_test)[:, 1]
print("Logistic ROC-AUC:", roc_auc_score(y_test, pred))


Logistic ROC-AUC: 0.7106177286289578


In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict_proba(X_test)[:, 1]
print("XGBoost ROC-AUC:", roc_auc_score(y_test, xgb_pred))


XGBoost ROC-AUC: 0.7227456989725065


In [ ]:
chunks = []
chunk_size = 200_000   # safe

for chunk in pd.read_csv(
    '/content/drive/MyDrive/rejected_loans.csv',
    low_memory=False,
    chunksize=chunk_size
):
    chunks.append(chunk)
    if len(chunks) == 5:   # 5 × 200k = 1M rows
        break

rej = pd.concat(chunks, ignore_index=True)
print(rej.shape)


(1000000, 9)


In [ ]:
accepted_reduced = accepted.copy()

# Create FICO proxy
accepted_reduced['fico_avg'] = (
    accepted_reduced['fico_range_low'] +
    accepted_reduced['fico_range_high']
) / 2

accepted_reduced = accepted_reduced[[
    'loan_amnt',
    'dti',
    'fico_avg',
    'emp_length',
    'addr_state',
    'default_flag'
]]

accepted_reduced.dropna(inplace=True)

accepted_reduced.shape


(1266782, 6)

In [ ]:
rej.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 9 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   Amount Requested      1000000 non-null  float64
 1   Application Date      1000000 non-null  object 
 2   Loan Title            999972 non-null   object 
 3   Risk_Score            773401 non-null   float64
 4   Debt-To-Income Ratio  1000000 non-null  object 
 5   Zip Code              999958 non-null   object 
 6   State                 999979 non-null   object 
 7   Employment Length     980283 non-null   object 
 8   Policy Code           1000000 non-null  float64
dtypes: float64(3), object(6)
memory usage: 68.7+ MB


In [ ]:
rej.head()


,Amount Requested,Application Date,Loan Title,Risk_Score,Debt-To-Income Ratio,Zip Code,State,Employment Length,Policy Code
0,1000.0,2007-05-26,Wedding Covered but No Honeymoon,693.0,10%,481xx,NM,4 years,0.0
1,1000.0,2007-05-26,Consolidating Debt,703.0,10%,010xx,MA,< 1 year,0.0
2,11000.0,2007-05-27,Want to consolidate my debt,715.0,10%,212xx,MD,1 year,0.0
3,6000.0,2007-05-27,waksman,698.0,38.64%,017xx,MA,< 1 year,0.0
4,1500.0,2007-05-27,mdrigo,509.0,9.43%,209xx,MD,< 1 year,0.0


In [ ]:
rej = rej.rename(columns={
    'Amount Requested': 'amount_requested',
    'Application Date': 'application_date',
    'Loan Title': 'loan_title',
    'Risk_Score': 'risk_score',
    'Debt-To-Income Ratio': 'dti',
    'Zip Code': 'zip_code',
    'State': 'state',
    'Employment Length': 'emp_length',
    'Policy Code': 'policy_code'
})


In [ ]:
rej.columns

Index(['amount_requested', 'application_date', 'loan_title', 'risk_score',
       'dti', 'zip_code', 'state', 'emp_length', 'policy_code'],
      dtype='object')

In [ ]:
rej['dti'] = (
    rej['dti']
    .str.replace('%', '', regex=False)
    .astype(float)
)

In [ ]:
rej['dti'].describe()

,dti
count,1.000000e+06
mean,8.306368e+02
std,5.226211e+04
min,-1.000000e+00
25%,5.570000e+00
50%,1.704000e+01
75%,3.146000e+01
max,5.000003e+07


In [ ]:
emp_map = {
    '< 1 year': 0.5,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10
}

rej['emp_length_num'] = rej['emp_length'].map(emp_map)

In [ ]:
def emp_to_num(x):
    if pd.isna(x): return 0
    x = str(x)
    if x == '10+ years': return 10
    if x == '< 1 year': return 0.5
    try:
        return float(x.split()[0])
    except:
        return 0

accepted_reduced['emp_length_num'] = accepted_reduced['emp_length'].apply(emp_to_num)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

numeric_cols = ['loan_amnt','dti','fico_avg','emp_length_num']

scaler = StandardScaler()
accepted_reduced[numeric_cols] = scaler.fit_transform(
    accepted_reduced[numeric_cols]
)

X = pd.get_dummies(
    accepted_reduced[numeric_cols + ['addr_state']],
    drop_first=True
)
y = accepted_reduced['default_flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

xgb_reduced = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=42
)

xgb_reduced.fit(X_train, y_train)

print(
    "ROC-AUC:",
    roc_auc_score(y_test, xgb_reduced.predict_proba(X_test)[:,1])
)


ROC-AUC: 0.6493000521522998


In [ ]:
chunks = []
for chunk in pd.read_csv(
    '/content/drive/MyDrive/rejected_loans.csv',
    low_memory=False,
    chunksize=200_000
):
    chunks.append(chunk)
    if len(chunks) == 5:
        break

rej = pd.concat(chunks, ignore_index=True)


In [ ]:
emp_map = {
    '< 1 year': 0.5, '1 year':1,'2 years':2,'3 years':3,'4 years':4,
    '5 years':5,'6 years':6,'7 years':7,'8 years':8,'9 years':9,'10+ years':10
}

rej['emp_length_num'] = rej['emp_length'].map(emp_map).fillna(0)
rej['dti'] = rej['dti'].str.replace('%','').astype(float).clip(0,65)

rej_reduced = rej[[
    'amount_requested','dti','risk_score','emp_length_num','state'
]].copy()

rej_reduced.rename(columns={
    'amount_requested':'loan_amnt',
    'risk_score':'fico_avg',
    'state':'addr_state',
    'emp_length_num':'emp_length'
}, inplace=True)


In [ ]:
rej_model = pd.get_dummies(rej_reduced, drop_first=True)

# align columns
rej_model = rej_model.reindex(columns=X.columns, fill_value=0)

# scale numeric
rej_model[numeric_cols] = scaler.transform(rej_model[numeric_cols])


In [ ]:
rej['default_prob'] = xgb_reduced.predict_proba(rej_model)[:,1]


In [ ]:
bins = [0,0.3,0.5,0.7,1.0]
labels = ['Low','Medium','High','Very High']

rej['risk_category'] = pd.cut(rej['default_prob'], bins=bins, labels=labels)

rej[['loan_title','default_prob','risk_category']].to_csv(
    '/content/drive/MyDrive/rejected_loans_with_predictions(reduced_features).csv',
    index=False
)


In [ ]:
import joblib

# paths
MODEL_PATH = '/content/drive/MyDrive/xgb_reduced_credit_model.pkl'
SCALER_PATH = '/content/drive/MyDrive/credit_scaler.pkl'
COLUMNS_PATH = '/content/drive/MyDrive/model_columns.pkl'

# save
joblib.dump(xgb_reduced, MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)
joblib.dump(X.columns.tolist(), COLUMNS_PATH)

print("✅ Model, scaler, and columns saved")


✅ Model, scaler, and columns saved


In [ ]:
rej[['emp_length', 'emp_length_num']].head(10)

,emp_length,emp_length_num
0,4 years,4.0
1,< 1 year,0.5
2,1 year,1.0
3,< 1 year,0.5
4,< 1 year,0.5
5,3 years,3.0
6,< 1 year,0.5
7,2 years,2.0
8,4 years,4.0
9,4 years,4.0


In [ ]:
rej['risk_score'] = rej['risk_score'].fillna(
    rej['risk_score'].median()
)

In [ ]:
rej_features = rej[
    ['amount_requested', 'dti', 'risk_score', 'emp_length_num', 'policy_code']
].copy()

In [ ]:
rej_features.isna().mean()

,0
amount_requested,0.000000
dti,0.000000
risk_score,0.000000
emp_length_num,0.019717
policy_code,0.000000


In [ ]:
# Remove invalid DTI values
rej['dti'] = rej['dti'].where(rej['dti'] >= 0)

# Cap DTI at 65 (industry standard)
rej['dti'] = rej['dti'].clip(upper=65)

In [ ]:
rej['dti'].describe()

,dti
count,966603.000000
mean,21.886231
std,18.576107
min,0.000000
25%,6.790000
50%,17.830000
75%,32.090000
max,65.000000


In [ ]:
rej['emp_length_num'] = rej['emp_length_num'].fillna(0)

In [ ]:
rej_features = rej[
    ['amount_requested', 'dti', 'risk_score', 'emp_length_num', 'policy_code']
].copy()

rej_features.isna().sum()

,0
amount_requested,0
dti,33397
risk_score,0
emp_length_num,0
policy_code,0


In [ ]:
rej['dti'] = rej['dti'].fillna(rej['dti'].median())


In [ ]:
rej_features = rej[
    ['amount_requested', 'dti', 'risk_score', 'emp_length_num', 'policy_code']
].copy()

rej_features.isna().sum()


,0
amount_requested,0
dti,0
risk_score,0
emp_length_num,0
policy_code,0


,0
amount_requested,0
dti,0
risk_score,0
emp_length_num,0
policy_code,0


In [ ]:
rej_model = rej_features.rename(columns={
    'amount_requested': 'loan_amnt',
    'risk_score': 'fico_avg'
})

In [ ]:
# Step 0: Clean column names
rej.columns = (
    rej.columns
    .str.strip()               # remove extra spaces
    .str.lower()               # lowercase everything
    .str.replace(" ", "_")     # replace spaces with underscores
)

print(rej.columns.tolist())


['amount_requested', 'application_date', 'loan_title', 'risk_score', 'dti', 'zip_code', 'state', 'emp_length', 'policy_code', 'emp_length_num']


In [ ]:
rej.columns = rej.columns.str.replace("-", "_")
print(rej.columns.tolist())
# Output should be: ['amount_requested', 'application_date', 'loan_title', 'risk_score', 'debt_to_income_ratio', 'zip_code', 'state', 'employment_length', 'policy_code']


['amount_requested', 'application_date', 'loan_title', 'risk_score', 'dti', 'zip_code', 'state', 'emp_length', 'policy_code', 'emp_length_num']


In [ ]:
# Clean and standardize column names
rej.columns = (
    rej.columns
    .str.strip()              # remove leading/trailing spaces
    .str.lower()              # lowercase everything
    .str.replace(" ", "_")    # replace spaces with underscores
    .str.replace("-", "_")    # replace dashes with underscores
)

# Verify
print(rej.columns.tolist())


['amount_requested', 'application_date', 'loan_title', 'risk_score', 'dti', 'zip_code', 'state', 'emp_length', 'policy_code', 'emp_length_num']


In [ ]:
# Select only the needed columns
rej_reduced = rej[[
    'amount_requested',   # loan amount
    'dti',                # debt-to-income ratio
    'risk_score',         # FICO proxy
    'emp_length_num',     # numeric employment length
    'state'               # state
]].copy()

# Rename columns to match accepted loans
rej_reduced.rename(columns={
    'amount_requested': 'loan_amnt',
    'dti': 'dti',
    'risk_score': 'fico_avg',
    'emp_length_num': 'emp_length',
    'state': 'addr_state'
}, inplace=True)

# Check
print(rej_reduced.head())
print(rej_reduced.shape)


   loan_amnt    dti  fico_avg  emp_length addr_state
0     1000.0  10.00     693.0         4.0         NM
1     1000.0  10.00     703.0         0.5         MA
2    11000.0  10.00     715.0         1.0         MD
3     6000.0  38.64     698.0         0.5         MA
4     1500.0   9.43     509.0         0.5         MD
(1000000, 5)


In [ ]:
print(rej_reduced.columns)

Index(['loan_amnt', 'dti', 'fico_avg', 'emp_length', 'addr_state'], dtype='object')


In [ ]:
rej_model = rej_reduced.copy()

In [ ]:
# 2️⃣ One-hot encode categorical columns (same as training)
rej_model = pd.get_dummies(rej_model, drop_first=True)


In [ ]:
# 1️⃣ Convert emp_length to numeric
def emp_to_num(x):
    if pd.isna(x):
        return 0
    x = str(x).strip()
    if x == '< 1 year':
        return 0.5
    elif x == '10+ years':
        return 10
    else:
        # extract leading number
        try:
            return float(x.split()[0])
        except:
            return 0

accepted_reduced['emp_length_num'] = accepted_reduced['emp_length'].apply(emp_to_num)

# 2️⃣ Choose numeric columns
numeric_cols = ['loan_amnt', 'dti', 'fico_avg', 'emp_length_num']

# 3️⃣ Scale numeric columns
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
accepted_reduced[numeric_cols] = scaler.fit_transform(accepted_reduced[numeric_cols])

# 4️⃣ Encode categorical columns
categorical_cols = ['addr_state']
X = pd.get_dummies(accepted_reduced[numeric_cols + categorical_cols], drop_first=True)
y = accepted_reduced['default_flag']

# 5️⃣ Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 6️⃣ Train XGBoost
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

xgb_reduced = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=42
)
xgb_reduced.fit(X_train, y_train)

print("ROC-AUC on reduced features:", roc_auc_score(y_test, xgb_reduced.predict_proba(X_test)[:,1]))


ROC-AUC on reduced features: 0.6493000521522998


In [ ]:
# Initialize column with NaN
rej['default_prob'] = np.nan

# Assign predictions to the rows that were used
rej.loc[rej_reduced.index, 'default_prob'] = rej_pred

# Check top 5
rej[['loan_title', 'default_prob']].head()


,loan_title,default_prob
0,Wedding Covered but No Honeymoon,0.115299
1,Consolidating Debt,0.089294
2,Want to consolidate my debt,0.123505
3,waksman,0.254053
4,mdrigo,0.148081


In [ ]:
rej.sort_values('default_prob', ascending=False)[['loan_title', 'default_prob']].head(10)


,loan_title,default_prob
995237,debt_consolidation,0.693171
889161,credit_card,0.693171
823769,credit_card,0.693171
830848,debt_consolidation,0.692369
863093,credit_card,0.683437
997802,debt_consolidation,0.674715
788385,credit_card,0.673817
765209,debt_consolidation,0.671094
921078,debt_consolidation,0.666462
897549,credit_card,0.665703


In [ ]:
bins = [0, 0.1, 0.2, 0.3, 1.0]
labels = ['Low', 'Medium', 'High', 'Very High']
rej['risk_category'] = pd.cut(rej['default_prob'], bins=bins, labels=labels)

rej['risk_category'].value_counts()


,count
risk_category,
High,360398
Medium,320814
Very High,299198
Low,19569


In [ ]:
rej[['loan_title', 'default_prob', 'risk_category']].to_csv(
    '/content/drive/MyDrive/rejected_loans_with_predictions.csv', index=False
)
